# 저가 / 고가 분리 모델 (LightGBM + Quantile Regression)
- 저가: ≤ 1,699만원 / 고가: ≥ 1,700만원
- CV TE 적용 완료 (누출 없음)

In [ ]:
# ── Step 1. 라이브러리 ────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error
import lightgbm as lgb

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

print('라이브러리 로드 완료')
print(f'  LightGBM  : {lgb.__version__}')

In [ ]:
# ── Step 2. 데이터 불러오기 ───────────────────────────────────
DATA_DIR = r'C:\Users\Admin\hipython\ml\bermuda.project\data'

low_train  = pd.read_csv(f'{DATA_DIR}/low_train.csv',  encoding='utf-8-sig')
low_test   = pd.read_csv(f'{DATA_DIR}/low_test.csv',   encoding='utf-8-sig')
high_train = pd.read_csv(f'{DATA_DIR}/high_train.csv', encoding='utf-8-sig')
high_test  = pd.read_csv(f'{DATA_DIR}/high_test.csv',  encoding='utf-8-sig')

print(f'저가 Train : {low_train.shape}  |  Test : {low_test.shape}')
print(f'고가 Train : {high_train.shape}  |  Test : {high_test.shape}')

In [ ]:
# ── Step 3. 피처 / 타겟 분리 ─────────────────────────────────
TARGET   = '현재가격_만원'
DROP_COLS = ['매물ID', '모델', '모델_원본']   # 비피처 컬럼

# 모델_LE : LightGBM categorical_feature로 지정 예정
CAT_COLS = ['모델_LE']

def make_Xy(df):
    X = df.drop(columns=[TARGET] + DROP_COLS)
    y = np.log1p(df[TARGET])   # 로그 변환
    return X, y

X_low_train,  y_low_train  = make_Xy(low_train)
X_low_test,   y_low_test   = make_Xy(low_test)
X_high_train, y_high_train = make_Xy(high_train)
X_high_test,  y_high_test  = make_Xy(high_test)

print('── 저가 ──')
print(f'  X_train: {X_low_train.shape}  y_train: {y_low_train.shape}')
print(f'  X_test : {X_low_test.shape}   y_test : {y_low_test.shape}')
print(f'  가격 범위 (원래): {np.expm1(y_low_train).min():.0f} ~ {np.expm1(y_low_train).max():.0f}만원')

print('\n── 고가 ──')
print(f'  X_train: {X_high_train.shape}  y_train: {y_high_train.shape}')
print(f'  X_test : {X_high_test.shape}   y_test : {y_high_test.shape}')
print(f'  가격 범위 (원래): {np.expm1(y_high_train).min():.0f} ~ {np.expm1(y_high_train).max():.0f}만원')

print('\n피처 목록:')
print(list(X_low_train.columns))